In [7]:
# duomenys 2010namuukiai.csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib tk
import warnings
warnings.simplefilter("ignore")

nukiai_2010 = pd.read_csv('2010namuukiai.csv')
bustas = nukiai_2010[['HH010', 'HH021', 'HH030', 'HH031', 'AP', 'SL', 'M_K', 'HH070', 'HY020']]

# pridedamos požymių reikšmės lengvesniam grafikų atvaizdavimui
mas = []
mas1 = []
for i in bustas['HH010'].values:
    if i < 3:
        mas.append(1)
        mas1.append('Namas')
    elif i > 4:
        mas.append(3)
        mas1.append('Kita')
    else:
        mas.append(2)
        mas1.append('Butas')
bustas['HH011'] = mas
bustas['HH012'] = mas1

mas = []
mas1 = []
for i in bustas['HH021'].values:
    if i < 3:
        mas.append(1)
        mas1.append('Savininkas')
    elif i > 4:
        mas.append(3)
        mas1.append('Be nuomos')
    else:
        mas.append(2)
        mas1.append('Nuoma')
bustas['HH022'] = mas
bustas['HH023'] = mas1

mas = []
for i in bustas['AP'].values:
    if i == 1:
        mas.append('Alytus')
    elif i == 2:
        mas.append('Kaunas')
    elif i == 3:
        mas.append('Klaipėda')
    elif i == 4:
        mas.append('Marijampolė')
    elif i == 5:
        mas.append('Panevėžys')
    elif i == 6:
        mas.append('Šiauliai')
    elif i == 7:
        mas.append('Tauragė')
    elif i == 8:
        mas.append('Telšiai')
    elif i == 9:
        mas.append('Utena')
    elif i == 10:
        mas.append('Vilnius')
bustas['AP_MST'] = mas

bustas_gr_tip = bustas.groupby(['HH011', 'HH022'])
bust_nuos_miest = bustas.groupby(['AP', 'HH011' ])


def bust_cnt(group, lentele):       #funkcija įrašų kiekiui suskaičiuoti pagal užsiduotą raktą
    sk = lentele.get_group(group)
    kiekis = sk['HH022'].value_counts()
    return kiekis

def surad_unik(objektas):       # funkcija, leidžianti braižant grafikus ašyje nenaudoti pasikartojančių reikšmių
                                # metoduose ticks ir ticklabels
      i = 0                     # grąžina: objetas - unikalių reikšmių masyvas
      j = 0                     # registras - pozicijų, į kurias rašomos reikšmės ašyje, masyvas
      registras = []
      pab  = True

      while pab:
            if objektas[i] == objektas[i + 1] and i + 1 < len(objektas):
                  if i == 0:
                        j += 1
                        registras.append(j)
                  objektas.pop(i)
                  j += 1
                  if i + 1 == len(objektas):
                        break
                  if objektas[i] == objektas[i + 1] and i + 1 < len(objektas):
                        objektas.pop(i)
                        j += 1                  
            if i + 1 < len(objektas) - 1:
                  i= i + 1
                  j += 1
                  registras.append(j)
            else:
                  pab = False
      return objektas, registras 

##### 1-as grafikas

lentele = bustas_gr_tip
groups = bustas_gr_tip.groups.keys()
colors = ['red', 'green', 'blue']

fig, g = plt.subplots()
i = 0                                    # spalvų masyvo indeksas
ind = 0                                  # indeksas, leidžiantis legendoje vieną kartą aprašyti spalvų masyvo reikšmes
x = 1
busto_sav = []                           # būsto pagal nuosavybę reikšmių masyvas

for group in groups:
    sk = bust_cnt(group, lentele)
    d = bustas_gr_tip.get_group(group)
    sav = d["HH012"].values
    tip = d["HH023"].values
    busto_sav.append(sav[0])
    if ind < len(colors):                
        g0 = g.bar(x, sk, color = colors[i], label = tip[0])
        g.bar_label(g0, fmt='%.2f')
        ind += 1
    else:
        g0 = g.bar(x, sk, color = colors[i])
        g.bar_label(g0, fmt='%.2f')
    x = x + 1
    i = i + 1
    if i > len(colors) - 1:
            i = 0

busto_sav, registras = surad_unik(busto_sav)
g.set_xticks(registras)
g.set_xticklabels(list(busto_sav), fontsize = 14)
g.set_xlabel('Būsto tipas', fontsize = 10, loc = 'right')
g.legend(title = 'Būsto nuosavybė')
fig.suptitle('Būsto tipo ir būsto pagal nuosavybę pasiskirstymas Lietuvoje 2010')
plt.show()


##### 2-as grafikas

lentele = bust_nuos_miest
groups = bust_nuos_miest.groups.keys()
colors = ['red', 'green', 'blue']
nuos = ['Namas', 'Butas', 'Kita']       # būsto pagal tipą reikšmių masyvas
miestas0 = []                           # didelių miestų masyvas
miestas1 = []                           # mažesnių miestų masyvas
x = 1
x1 = 1
i = 0                                   # spalvų masyvo indeksas
i1 = 0
j = 0                                   # miestų masyvo indeksas
j1 = 0
ind = 0                                 # indeksas, leidžiantis legendoje vieną kartą aprašyti spalvų masyvo reikšmes
x = 1

fig, g = plt.subplots(1,2)

for group in groups:    
    sk = bust_cnt(group, lentele)
    ad = bust_nuos_miest.get_group(group)
    mst = ad['AP_MST'].values
    if (group[0] > 1 and group[0] < 4) or (group[0] > 4 and group[0] < 7) or group[0] == 10:       
        miestas0.append(mst[0])
        if i ==0 or miestas0[j] == miestas0[j - 1]:     # ar tai kitas miestas, kad spalvų masyve imti pirmąją reikšmę
            g[0].bar(x, sk, color = colors[i])          
        else:
            i = 0
            g[0].bar(x, sk, color = colors[i])           
        j += 1
        x = x + 1
        i = i + 1
        if i > len(colors) - 1:
            i = 0
    else:                                               # jei tai mažesnis miestas
        miestas1.append(mst[0])        
        if ind < len(colors):
            g[1].bar(x1, sk, color = colors[i1], label = nuos[i1])
            ind = ind + 1
        elif miestas1[j1] == miestas1[j1 -1]:
            g[1].bar(x1, sk, color = colors[i1])
        else:
            i1 = 0
            g[1].bar(x1, sk, color = colors[i1])                    
        x1 = x1 + 1
        i1 = i1 + 1
        j1 += 1
        if i1 > len(colors) - 1:
            i1 = 0

g[0].set_ylabel('Būstų kiekis', fontsize = 14)
g[0].set_title('Didieji miestai')
g[0].set_ylim(0,1000)
g[0].set_yticks(np.arange(0, 1001, 200))
miest_0, registras0 = surad_unik(miestas0)
g[0].set_xticks(registras0)
g[0].set_xticklabels(list(miest_0), rotation = 90)

g[1].set_title('Mažesni miestai')
miest_1, registras1 = surad_unik(miestas1)
g[1].set_xticks(registras1)
g[1].set_xticklabels(list(miest_1), rotation = 90,)
g[1].legend(title ='Būsto tipas')
g[1].set_ylim(0,1000)
g[1].set_yticks(np.arange(0, 1001, 200))

fig.suptitle('Būstų pasiskirstymas pagal tipą Lietuvoje 2010')
plt.show()


##### 3-as grafikas

lentele = bust_nuos_miest
groups = bust_nuos_miest.groups.keys()
colors = ['red', 'green', 'blue']
nuos = ['Namas', 'Butas', 'Kita']           # būsto pagal tipą reikšmių masyvas
miestas0 = []                               # didelių miestų masyvas
miestas1 = []                               # mažesnių miestų masyvas
bust_mok_vid_0 = []                         # dideliuose miestuose mokamų mokesčių vidurkių masyvas
bust_mok_vid_1 = []                         # mažesniuose miestuose mokamų mokesčių vidurkių masyvas
x = 1
x1 = 1
i = 0                                       # spalvų masyvo indeksas
i1 = 0
j = 0                                       # miestų masyvo indeksas
j1 = 0
ind = 0                                     # indeksas, leidžiantis legendoje vieną kartą aprašyti spalvų masyvo reikšmes

fig, g = plt.subplots(1,2)

for group in groups:    
    sk = bust_cnt(group, lentele) 
    ad = bust_nuos_miest.get_group(group)
    mst = ad['AP_MST'].values
    if (group[0] > 1 and group[0] < 4) or (group[0] > 4 and group[0] < 7) or group[0] == 10:       
        miestas0.append(mst[0])
        d_0 = bust_nuos_miest.get_group(group)
        bust_mok_0 = d_0["HH070"].values
        bust_mok_vid_0.append(round(np.average(bust_mok_0),2))        
        if i ==0 or miestas0[j] == miestas0[j - 1]:         # ar tai kitas miestas, kad spalvų masyve imti pirmąją reikšmę
            g[0].bar(x, sk, color = colors[i])            
        else:
            i = 0
            g[0].bar(x, sk, color = colors[i]) 
        j += 1
        x = x + 1
        i = i + 1
        if i > len(colors) - 1:
            i = 0
    else:                                   # jei tai mažesnis miestas
        miestas1.append(mst[0])
        d_1 = bust_nuos_miest.get_group(group)
        bust_mok_1 = d_1["HH070"].values
        bust_mok_vid_1.append(round(np.average(bust_mok_1),2))        
        if ind < len(colors):
            g[1].bar(x1, sk, color = colors[i1], label = nuos[i1])
            ind = ind + 1
        elif miestas1[j1] == miestas1[j1 -1]:
            g[1].bar(x1, sk, color = colors[i1])
        else:
            i1 = 0
            g[1].bar(x1, sk, color = colors[i1])
                    
        x1 = x1 + 1
        i1 = i1 + 1
        j1 += 1
        if i1 > len(colors) - 1:
            i1 = 0

g[0].set_ylabel('Būstų kiekis', fontsize = 14)
g[0].set_title('Didieji miestai')
g[0].set_ylim(0,1000)
g[0].set_yticks(np.arange(0, 1001, 200))
x_zm_0 = np.arange(1, len(bust_mok_vid_0) + 1)
g[0].plot(x_zm_0, bust_mok_vid_0, ls = '-', lw = 2, color = "brown", marker='o', ms = 4)
miest_0, registras0 = surad_unik(miestas0)
g[0].set_xticks(registras0)
g[0].set_xticklabels(list(miest_0), rotation = 90)

g[1].set_title('Mažesni miestai')
miest_1, registras1 = surad_unik(miestas1)
x_zm_1 = np.arange(1, len(bust_mok_vid_1) + 1)
g[1].plot(x_zm_1, bust_mok_vid_1, ls = '-', lw = 2, color = "brown", marker='o', ms = 4, label = 'Būsto mokesčiai')
g[1].set_xticks(registras1)
g[1].set_xticklabels(list(miest_1), rotation = 90,)
g[1].legend()
g[1].set_ylim(0,1000)
g[1].set_yticks(np.arange(0, 1001, 200))

fig.suptitle('Namų ūkių mokesčiai už būstą Lietuvoje 2010')
plt.show()
